# Experiment workflow

In [8]:
import random
import pandas as pd
from pathlib import Path
import json
import numpy as np

DATA_DIR = Path("/home/wzhan969/robustness-misinfo/data/dataset")
SEED = 42

1. Sample the claims

In [9]:
ds_bias = pd.read_csv(DATA_DIR / "ds_bias.csv")
ds_conspiracy = pd.read_csv(DATA_DIR / "ds_conspiracy.csv")

In [10]:
# Number of unique values in the "bias_type" column
unique_bias_type_count = ds_bias["bias_type"].nunique()
print(f"Unique bias_type values: {unique_bias_type_count}")

# Frequency of each unique value
print(ds_bias["bias_type"].value_counts(dropna=False))

Unique bias_type values: 13
bias_type
['gender']                        23
['regional-person']               14
['physical-appearance']            8
['gender+age']                     6
['age']                            6
['political']                      3
['urbanity']                       3
['ethnicity']                      2
['disability']                     2
['region']                         2
['gender+physical-appearance']     1
['sexual-orientation']             1
['regional-person+gender']         1
Name: count, dtype: int64


In [11]:
# Number of unique values in the "type" column
unique_conspiracy_type_count = ds_conspiracy["type"].nunique()
print(f"Unique conspiracy_type values: {unique_conspiracy_type_count}")

# Frequency of each unique value
print(ds_conspiracy["type"].value_counts(dropna=False))

Unique conspiracy_type values: 5
type
government malfeasance          16
personal wellbeing              16
malevolent global conspiracy     9
control of information           8
extraterrestrial cover-up        7
NaN                              3
Name: count, dtype: int64


In [12]:
def largest_remainder_allocation(counts: pd.Series, n: int) -> dict:
    """
    Allocate exactly n samples across categories proportional to counts,
    using the largest remainder method to resolve rounding.

    Parameters
    ----------
    counts : pd.Series
        Index = category labels, values = population counts.
    n : int
        Total number of samples to allocate.

    Returns
    -------
    dict  {category: n_samples}
    """
    total = counts.sum()
    exact = counts * n / total

    # Floor allocation
    floor_alloc = np.floor(exact).astype(int)
    remainder = n - floor_alloc.sum()

    # Distribute remaining slots to categories with largest fractional parts
    fractional = exact - floor_alloc
    top_up = fractional.nlargest(remainder).index
    floor_alloc[top_up] += 1

    # Ensure every category with count >= 1 gets at least 1 if possible
    # (already handled by largest-remainder for large-enough n)
    return floor_alloc.to_dict()

In [13]:
def sample_bias(path: Path, n: int, seed: int) -> pd.DataFrame:
    df = pd.read_csv(path)

    counts = df["bias_type"].value_counts()
    allocation = largest_remainder_allocation(counts, n)

    print("\nBias allocation:")
    for cat, k in sorted(allocation.items(), key=lambda x: -x[1]):
        print(f"  {str(cat):<35} population={counts[cat]:>2}  sample={k}")

    frames = []
    for cat, k in allocation.items():
        if k > 0:
            frames.append(
                df[df["bias_type"] == cat].sample(n=k, random_state=seed)
            )

    return pd.concat(frames, ignore_index=True)


def sample_conspiracy(path: Path, n: int, seed: int) -> pd.DataFrame:
    df = pd.read_csv(path)

    df_valid = df.dropna(subset=["type"]).copy()
    n_dropped = len(df) - len(df_valid)
    if n_dropped:
        print(f"\n  [conspiracy] dropped {n_dropped} rows with NaN type")

    counts = df_valid["type"].value_counts()
    allocation = largest_remainder_allocation(counts, n)

    print("\nConspiracy allocation:")
    for cat, k in sorted(allocation.items(), key=lambda x: -x[1]):
        print(f"  {str(cat):<35} population={counts[cat]:>2}  sample={k}")

    frames = []
    for cat, k in allocation.items():
        if k > 0:
            frames.append(
                df_valid[df_valid["type"] == cat].sample(n=k, random_state=seed)
            )

    return pd.concat(frames, ignore_index=True)

In [14]:
def build_claim_list(bias_sample: pd.DataFrame,
                     conspiracy_sample: pd.DataFrame) -> list[dict]:
    """Convert sampled rows into the claim-dict format used by run_session."""
    claims = []

    for _, row in bias_sample.iterrows():
        claims.append({
            "category": "bias",
            "content": row["content"],
            "subtype": row["bias_type"],
        })

    for _, row in conspiracy_sample.iterrows():
        claims.append({
            "category": "conspiracy",
            "content": row["content"],
            "subtype": row["type"],
        })

    return claims

In [15]:
N_PER_DATASET = 15

bias_sample = sample_bias(DATA_DIR / "ds_bias.csv", N_PER_DATASET, SEED)
conspiracy_sample = sample_conspiracy(DATA_DIR / "ds_conspiracy.csv", N_PER_DATASET, SEED)

claims = build_claim_list(bias_sample, conspiracy_sample)

print(f"\nTotal claims: {len(claims)}  "
        f"({sum(1 for c in claims if c['category']=='bias')} bias + "
        f"{sum(1 for c in claims if c['category']=='conspiracy')} conspiracy)")
print("\nFull claim list:")
for i, c in enumerate(claims):
    print(f"  [{i:02d}] ({c['category']:<10} | {str(c['subtype']):<35}) {c['content'][:70]}")


Bias allocation:
  ['gender']                          population=23  sample=5
  ['regional-person']                 population=14  sample=3
  ['physical-appearance']             population= 8  sample=2
  ['gender+age']                      population= 6  sample=1
  ['age']                             population= 6  sample=1
  ['political']                       population= 3  sample=1
  ['urbanity']                        population= 3  sample=1
  ['ethnicity']                       population= 2  sample=1
  ['disability']                      population= 2  sample=0
  ['region']                          population= 2  sample=0
  ['gender+physical-appearance']      population= 1  sample=0
  ['sexual-orientation']              population= 1  sample=0
  ['regional-person+gender']          population= 1  sample=0

  [conspiracy] dropped 3 rows with NaN type

Conspiracy allocation:
  government malfeasance              population=16  sample=4
  personal wellbeing                  populat

In [17]:
output_path = DATA_DIR / "sampled_claims.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(claims, f, ensure_ascii=False, indent=2)

print(f"Saved {len(claims)} claims to {output_path}")

Saved 30 claims to /home/wzhan969/robustness-misinfo/data/dataset/sampled_claims.json
